# Amazon ML Challenge 2026 — Business Entity Resolution
## Main Experiment + Integration Notebook

This notebook is the team's single research/integration notebook.

**Important:** keep the official `student_resource/` folder untouched. This notebook reads it;
it does not modify the challenge TSVs or the official validator.

## 0. Project overview

**The problem.** Every Source-1 (S1) business entity must be matched to **zero, one, or many**
corresponding Source-2/Source-3 (S2/S3) entities. This is a **multi-match** problem, not a
1:1 matching problem — a correct prediction for one S1 entity can be an empty set (a genuine
singleton, no real counterpart exists in S2/S3), a single ID, or several IDs.

**The metric is macro F0.5**, averaged per S1 entity. F0.5 weights **precision over recall**
(`beta=0.5`), so confidently wrong extra matches are punished harder than a missed match — the
macher should lean conservative rather than greedy. A correct singleton (empty ground truth,
empty prediction) scores a perfect 1.0; predicting anything for a true singleton scores 0.0 for
that entity.

**Blocking is a recall gate, not a modeling step.** Before any pairwise model can be trained,
Sudarshan's blocking stage must first propose a *candidate set* of plausible S2/S3 IDs for each
S1 entity. Nothing the model sees will ever include a true match that blocking dropped — so
blocking's **candidate recall** puts a hard ceiling on the final macro F0.5, independent of how
good the downstream classifier is. That is exactly what Part 4 below (and
`src/analyze_candidates.py`) measures.

### Project flow

```
DATA → NORMALIZATION → BLOCKING → CANDIDATE ANALYSIS → LABELING → FEATURES
     → MODEL → THRESHOLD → EVALUATION → ERROR ANALYSIS → TEST INFERENCE → SUBMISSION
```

### Current status

| Stage                 | Status                          |
|-----------------------|----------------------------------|
| Normalization          | COMPLETE (`src/normalization.py`, unit-tested; full train/test files pre-normalized in Drive) |
| Evaluation             | COMPLETE (`src/evaluation.py`, unit-tested) |
| Tests                  | COMPLETE (see `tests/`) |
| Candidate analysis     | READY (`src/analyze_candidates.py`, this notebook's Part 4) |
| Blocking output        | **WAITING FOR SUDARSHAN** (real `candidate_pairs.tsv` not yet delivered) |
| ML baseline            | **WAITING FOR REAL CANDIDATES** (labeling/features/model sections below run automatically once the file lands) |

No candidate recall or model performance numbers are claimed here until the real candidate file
exists — every number below is computed live from whatever files are actually present, and every
stage that depends on a file that doesn't exist yet **skips itself with a clear message** instead
of fabricating results or crashing.

**Feature generation note.** `src/features.py::pair_features` (the original, unit-tested B1-B6
implementation) takes RAW `business_name`/`business_address`/`country` and normalizes internally
on every call. That is fine for a handful of pairs, but re-running Unicode normalization per pair
across millions of real candidate pairs is wasteful once a normalized data root already exists.
Part 7 below therefore uses the newer, additive `pair_features_normalized` /
`build_pair_feature_frame_normalized_chunked` path (`src/features.py`), which consumes the
precomputed `name_norm`/`address_norm`/`country_norm` columns and processes candidate pairs in
bounded chunks. `pair_features` itself is untouched — see its docstring and
`tests/test_normalized_features.py` for the equivalence proof (and one documented, intentional
exception around country-alias spellings).

In [ ]:
# ================================
# 0.1 ENVIRONMENT BOOTSTRAP (Colab + local fallback)
# ================================
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    PROJECT_ROOT = Path("/content/amazon_ml_challenge_2026")
    if not PROJECT_ROOT.exists():
        !git clone https://github.com/Mirunjai/amazon_ml_challenge_2026.git {PROJECT_ROOT}
    !pip install -q -r {PROJECT_ROOT}/requirements.txt

    # Shared team dataset location.
    STUDENT_RESOURCE = Path("/content/drive/MyDrive/student_resource")
else:
    # Local / non-Colab fallback: walk up from the current directory until we
    # find the repo root (the folder containing `src/`), and assume the
    # official dataset lives in a sibling `student_resource/` folder.
    PROJECT_ROOT = Path.cwd()
    while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent
    STUDENT_RESOURCE = PROJECT_ROOT / "student_resource"

sys.path.insert(0, str(PROJECT_ROOT))

print("Environment      :", "Colab" if IN_COLAB else "Local")
print("Project root     :", PROJECT_ROOT, "->", "OK" if PROJECT_ROOT.exists() else "MISSING")
print("Dataset root     :", STUDENT_RESOURCE, "->", "OK" if STUDENT_RESOURCE.exists() else "MISSING")

## 1. Configuration

All paths are centralized here and nowhere else in the notebook. `ChallengePaths`
(`src/config.py`) is reused rather than re-deriving `dataset/train` / `dataset/test` paths by
hand.

**Normalized data root.** The team's full train/test Source-1/2/3 files are already normalized
(they carry `name_norm`, `name_nosuffix`, `address_norm`, `country_norm` alongside the original
raw columns — the exact shape `src.normalization.add_normalized_columns` produces). In Colab
these live in Google Drive; this is the one place that path is set. Locally (no Drive mounted)
the notebook falls back to the same relative layout under `student_resource/`, and Section 6
normalizes Source-1 on demand if the normalized columns aren't actually present there yet.

The additional derived-artifact paths below (candidate files, experiment log) extend the same
convention already used by the repo's `.gitignore` (`outputs/candidates/`, `outputs/experiments/`,
`outputs/final/`) — they are our own derived artifacts, never written under `student_resource/`.

In [ ]:
# =========================
# 1.1 Imports
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# =========================
# 1.2 Centralized paths
# =========================
from src.config import ChallengePaths, REQUIRED_COLUMNS, GROUND_TRUTH_COLUMNS, CANDIDATE_COLUMNS

PATHS = ChallengePaths.from_student_resource(
    STUDENT_RESOURCE,
    output_dir=PROJECT_ROOT / "outputs" / "final",
)

# --- normalized data root (THE one centralized place this path is set) ---
if IN_COLAB:
    NORMALIZED_ROOT = Path("/content/drive/MyDrive/student_resource/dataset/normalized")
else:
    # Local fallback: same relative layout, in case a normalized/ folder has
    # been staged next to dataset/train and dataset/test.
    NORMALIZED_ROOT = STUDENT_RESOURCE / "dataset" / "normalized"

# --- official challenge files ---
# Train/test Source-1/2/3 are read from the normalized root above (see Section 1
# markdown); ground truth and the official validator stay under student_resource/
# exactly as provided (nothing to normalize there).
TRAIN_S1 = NORMALIZED_ROOT / "train_source1.tsv"
TRAIN_S2 = NORMALIZED_ROOT / "train_source2.tsv"
TRAIN_S3 = NORMALIZED_ROOT / "train_source3.tsv"
GROUND_TRUTH = PATHS.train_file("train_ground_truth.tsv")

TEST_S1 = NORMALIZED_ROOT / "test_source1.tsv"
TEST_S2 = NORMALIZED_ROOT / "test_source2.tsv"
TEST_S3 = NORMALIZED_ROOT / "test_source3.tsv"

VALIDATOR_PATH = STUDENT_RESOURCE / "utils" / "validate_submission.py"

# --- our own derived artifacts (never written into student_resource/) ---
CANDIDATE_PATH = PROJECT_ROOT / "outputs" / "candidates" / "candidate_pairs.tsv"       # Sudarshan's TRAIN blocking output
TEST_CANDIDATE_PATH = PROJECT_ROOT / "outputs" / "candidates" / "test_candidate_pairs.tsv"  # blocking output on the TEST set
OUTPUT_DIR = PATHS.output_dir                                                          # outputs/final -> matching_results.tsv, candidate_pairs.tsv
EXPERIMENT_LOG_PATH = PROJECT_ROOT / "outputs" / "experiments" / "experiment_log.csv"

# Chunk size used everywhere candidate pairs are streamed/featurized at scale
# (Parts 5, 7, 13). Centralized here so every stage agrees on one value.
CANDIDATE_CHUNKSIZE = 200_000

_ALL_PATHS = {
    "normalized data root": NORMALIZED_ROOT,
    "train_source1 (normalized)": TRAIN_S1,
    "train_source2 (normalized)": TRAIN_S2,
    "train_source3 (normalized)": TRAIN_S3,
    "train_ground_truth": GROUND_TRUTH,
    "test_source1 (normalized)": TEST_S1,
    "test_source2 (normalized)": TEST_S2,
    "test_source3 (normalized)": TEST_S3,
    "official validator": VALIDATOR_PATH,
    "train candidate_pairs (Sudarshan)": CANDIDATE_PATH,
    "test candidate_pairs": TEST_CANDIDATE_PATH,
}

print("Path configuration:")
for label, path in _ALL_PATHS.items():
    status = "OK   " if path.exists() else "MISS "
    print(f"  {status} {label:<34} {path}")

## 2. Dataset sanity checks

Lightweight checks only: confirm required files exist, confirm required columns exist, and
confirm IDs are treated as strings — without ever loading a multi-million-row table whole just
to check its shape. Source 1 and ground truth are modest in size and are loaded fully (they're
needed fully in later sections anyway); Source 2 / Source 3 and every test-side table are only
row-counted and sampled.

In [ ]:
# =========================
# 2.1 Existence + schema checks (sample-based for the large tables)
# =========================
from src.io_utils import read_tsv, count_tsv_rows, validate_columns

TRAIN_S1_AVAILABLE = TRAIN_S1.exists()
GROUND_TRUTH_AVAILABLE = GROUND_TRUTH.exists()

s1 = None
gt = None

if TRAIN_S1_AVAILABLE:
    s1 = read_tsv(TRAIN_S1, dtype=str, keep_default_na=False)
    validate_columns(s1.columns, REQUIRED_COLUMNS, context="train_source1")
    print(f"train_source1: {len(s1):,} rows, columns OK, entity_id dtype = {s1['entity_id'].dtype}")
else:
    print("train_source1: MISSING -- skipping Source-1 load.")

if GROUND_TRUTH_AVAILABLE:
    gt = read_tsv(GROUND_TRUTH, dtype=str, keep_default_na=False)
    validate_columns(gt.columns, GROUND_TRUTH_COLUMNS, context="train_ground_truth")
    print(f"train_ground_truth: {len(gt):,} rows, columns OK")
else:
    print("train_ground_truth: MISSING -- skipping ground-truth load.")

print()
print("Large tables (row count + column check only, no full load):")
for name, path in [
    ("train_source2", TRAIN_S2),
    ("train_source3", TRAIN_S3),
    ("test_source1", TEST_S1),
    ("test_source2", TEST_S2),
    ("test_source3", TEST_S3),
]:
    if not path.exists():
        print(f"  {name:<14} MISSING")
        continue
    sample = read_tsv(path, dtype=str, keep_default_na=False, nrows=5)
    validate_columns(sample.columns, REQUIRED_COLUMNS, context=name)
    print(f"  {name:<14} {count_tsv_rows(path):,} rows, columns OK (sampled, not fully loaded)")

In [ ]:
# =========================
# 2.2 Normalized-column presence check
# =========================
NORMALIZED_COLUMNS = ["name_norm", "name_nosuffix", "address_norm", "country_norm"]

if TRAIN_S1_AVAILABLE:
    present = [c for c in NORMALIZED_COLUMNS if c in s1.columns]
    missing = [c for c in NORMALIZED_COLUMNS if c not in s1.columns]
    if not missing:
        print(f"train_source1 already carries normalized columns: {present}")
    else:
        print(
            f"train_source1 (loaded from {TRAIN_S1}) is missing normalized columns {missing}. "
            "Section 6 will normalize Source-1 in-memory on demand instead "
            "(Source-1 alone, never the full Source-2/3 tables)."
        )
else:
    print("train_source1 is missing -- skipping normalized-column check.")

## 3. Ground truth sanity check

Reuses `src.evaluation.parse_ground_truth` and `src.evaluation.evaluate` -- no metric logic is
duplicated here. `evaluate(gt, gt)` must score a perfect identity: `macro_f0.5 == 1.0`.

In [ ]:
# =========================
# 3.1 Ground-truth identity check
# =========================
from src.evaluation import parse_ground_truth, evaluate

gt_map = None

if GROUND_TRUTH_AVAILABLE:
    gt_map = parse_ground_truth(GROUND_TRUTH)

    identity = evaluate(gt_map, gt_map)
    assert identity["macro_f0.5"] == 1.0, "Ground truth failed to score a perfect identity match!"
    print("Identity check macro_f0.5:", identity["macro_f0.5"], "(expected 1.0) -- PASSED")

    match_counts = pd.Series({k: len(v) for k, v in gt_map.items()}, name="match_count")
    print()
    print("S1 entities in ground truth :", f"{len(gt_map):,}")
    print("Unmatched (singleton) S1s   :", f"{int((match_counts == 0).sum()):,}")
    print("Total true matches          :", f"{int(match_counts.sum()):,}")
    print()
    print("Match-count distribution:")
    display(match_counts.value_counts().sort_index().to_frame("s1_entities"))
else:
    print("train_ground_truth.tsv is missing -- skipping ground-truth sanity check.")

## 4. Candidate analysis (blocking quality)

Integrates `src/analyze_candidates.py` directly (no shelling out, no duplicated logic). It
streams the candidate file in chunks internally (`src.io_utils.iter_tsv`), so this stays usable
on a multi-million-row file. This is BLOCKING candidate recall -- the ceiling on everything
downstream -- not model recall. If Sudarshan's candidate file doesn't exist yet, this stage
prints a clear waiting message and skips without crashing the rest of the notebook.

In [ ]:
# =========================
# 4.1 Candidate quality report
# =========================
from src.analyze_candidates import compute_report, format_report_text

CANDIDATES_AVAILABLE = CANDIDATE_PATH.exists() and GROUND_TRUTH_AVAILABLE
candidate_report = None

if CANDIDATE_PATH.exists() and not GROUND_TRUTH_AVAILABLE:
    print("Candidate file is present but train_ground_truth.tsv is missing -- cannot compute recall.")
elif not CANDIDATE_PATH.exists():
    print(
        "WAITING FOR SUDARSHAN'S CANDIDATE TSV.\n"
        f"Expected at: {CANDIDATE_PATH}\n"
        "Skipping candidate analysis, labeling, features, split, model, threshold, "
        "error analysis, and test inference -- all of that resumes automatically "
        "once this file exists."
    )
else:
    candidate_report = compute_report(CANDIDATE_PATH, GROUND_TRUTH, chunksize=CANDIDATE_CHUNKSIZE)
    print(format_report_text(candidate_report))

## 5. Candidate labeling

Uses `src.labeling` to turn Sudarshan's wide candidate file into labeled (S1, candidate) pairs
against the training ground truth -- 1 = true match, 0 = non-match. This labels only the pairs
that blocking actually proposed; it does not perform a random pair split (that comes later, at
the entity level, in Part 8).

The candidate file is one row per S1 entity, but it may still be extremely large, so it is never
loaded whole with a single `read_tsv`. It is streamed in chunks
(`src.io_utils.iter_tsv`) and each chunk is exploded into long-format pairs
(`src.labeling.explode_candidate_pairs`) immediately, so only the (much smaller) long-format
pieces accumulate in memory -- the full wide file is never materialized at once.

**Remaining scalability note:** the exploded long-format pairs from every chunk are still
concatenated into one in-memory DataFrame below (two string columns, one row per candidate pair)
because Part 7's chunked feature builder and Part 8's entity split both need pair-level
granularity over the whole set. For a genuinely huge candidate file (tens of millions of pairs)
this exploded frame -- not the original wide file -- would become the memory bottleneck; see the
final checklist / notebook summary for this as an open item rather than something silently solved
here.

In [ ]:
# =========================
# 5.1 Label candidate pairs
# =========================
from src.io_utils import iter_tsv
from src.labeling import explode_candidate_pairs, label_candidate_pairs

def _load_candidate_pairs_long(path, chunksize=CANDIDATE_CHUNKSIZE):
    '''Stream a wide candidate_pairs.tsv-shaped file (one row per S1 entity)
    in chunks and explode each chunk into long-format (s1, candidate) pairs
    immediately, so the full wide file is never held in memory at once --
    only the accumulated long-format pairs are (see the scalability note in
    the Part 5 markdown above: this is a reasoned, documented tradeoff, not
    an oversight).'''
    parts = [
        explode_candidate_pairs(chunk)
        for chunk in iter_tsv(path, chunksize=chunksize, dtype=str, keep_default_na=False)
    ]
    return (
        pd.concat(parts, ignore_index=True)
        if parts
        else pd.DataFrame(columns=["source1_entity_id", "candidate_entity_id"])
    )

long_pairs = None
label_df = None

if CANDIDATES_AVAILABLE:
    long_pairs = _load_candidate_pairs_long(CANDIDATE_PATH)
    label_df = label_candidate_pairs(long_pairs, gt_map)

    positives = int((label_df["label"] == 1).sum())
    negatives = int((label_df["label"] == 0).sum())
    total = len(label_df)

    print(f"Total candidate pairs : {total:,}")
    print(f"Positives (label=1)   : {positives:,}")
    print(f"Negatives (label=0)   : {negatives:,}")
    print(f"Positive rate         : {positives / total:.4%}" if total else "Positive rate: n/a (no pairs)")
else:
    print("No real candidate file yet -- skipping candidate labeling.")

## 6. Normalized data

Uses the already-normalized fields (`name_norm`, `name_nosuffix`, `address_norm`,
`country_norm`) wherever possible instead of re-normalizing millions of strings. Since Source-1
was already loaded from the normalized root in Section 2, if it already carries these columns
they are used directly; only if they're actually missing (e.g. a local run without the normalized
files staged yet) is Source-1 -- a bounded table, not a multi-million-row one -- normalized once
in-memory via `src.normalization.add_normalized_columns`. Source-2/Source-3 are deliberately
**not** normalized in bulk here -- Part 7 only ever fetches the small subset of S2/S3 rows that
actually appear as candidates (and normalizes that small subset if needed), once a real candidate
file exists.

In [ ]:
# =========================
# 6.1 Source-1 normalized columns
# =========================
from src.normalization import add_normalized_columns

s1_norm = None

if TRAIN_S1_AVAILABLE:
    missing_norm_cols = [c for c in NORMALIZED_COLUMNS if c not in s1.columns]
    if not missing_norm_cols:
        s1_norm = s1
        print(f"Using the already-normalized train_source1 columns loaded from {TRAIN_S1}.")
    else:
        s1_norm = add_normalized_columns(s1)
        print(f"Normalized Source-1 in-memory: {len(s1_norm):,} rows (not cached to disk).")
else:
    print("train_source1.tsv is missing -- skipping normalization.")

if s1_norm is not None:
    display(s1_norm[["entity_id", "name_norm", "name_nosuffix", "address_norm", "country_norm"]].head(5))

## 7. Baseline features

The baseline feature set is exactly the eight B1-B6 columns already implemented in
`src/features.py` -- no experimental features are added here. Only the S2/S3 rows actually
referenced as candidates are loaded (streamed + filtered), never the full multi-million-row
tables.

**Pre-normalized, chunked path.** `src/features.py::pair_features` (the original function) takes
RAW `business_name`/`business_address`/`country` and re-normalizes them on every call -- correct,
but at millions of candidate pairs that means repeating expensive Unicode-aware normalization
work that Section 6 (and the normalized data root itself) already did once. This section instead
uses the additive `pair_features_normalized` path via
`src.features.build_pair_feature_frame_normalized_chunked`, which:

- consumes the precomputed `name_norm` / `address_norm` / `country_norm` columns instead of
  re-normalizing text per pair, and
- processes `long_pairs` in bounded chunks (`CANDIDATE_CHUNKSIZE` rows at a time) rather than
  building one Python list of `(Series, Series)` tuples for every candidate pair up front -- the
  chunked helper looks up only `CANDIDATE_CHUNKSIZE` rows at a time, so peak memory does not grow
  with the total number of candidate pairs.

`pair_features` itself is unmodified (see its docstring, and `tests/test_baseline_features.py`
which still passes unchanged); `pair_features_normalized` is proven to give identical B1-B6
values on representative pairs in `tests/test_normalized_features.py`, with one documented,
intentional exception for country values that hit `COUNTRY_ALIASES` resolution differently than
the raw path's simpler comparison (see that function's docstring for the exact boundary).

In [ ]:
# =========================
# 7.1 Build baseline pairwise features for the real candidate pairs
#     (pre-normalized + chunked production path -- see Part 7 markdown)
# =========================
from src.io_utils import iter_tsv
from src.normalization import add_normalized_columns
from src.features import build_pair_feature_frame_normalized_chunked

feature_df = None
candidate_lookup = None

def _load_referenced_rows(path, needed_ids, chunksize=CANDIDATE_CHUNKSIZE):
    # Stream a Source-2/3 file, keeping only rows whose entity_id is needed.
    # Avoids loading the full (potentially multi-million-row) table just to
    # fetch business fields for a much smaller set of candidate IDs.
    if not needed_ids or not path.exists():
        return pd.DataFrame(columns=REQUIRED_COLUMNS)
    keep = []
    for chunk in iter_tsv(path, chunksize=chunksize, dtype=str, keep_default_na=False):
        keep.append(chunk[chunk["entity_id"].isin(needed_ids)])
    return pd.concat(keep, ignore_index=True) if keep else pd.DataFrame(columns=REQUIRED_COLUMNS)

if CANDIDATES_AVAILABLE and long_pairs is not None and TRAIN_S1_AVAILABLE:
    needed_candidate_ids = set(long_pairs["candidate_entity_id"])

    s2_referenced = _load_referenced_rows(TRAIN_S2, needed_candidate_ids)
    s3_referenced = _load_referenced_rows(TRAIN_S3, needed_candidate_ids)
    candidate_lookup = pd.concat([s2_referenced, s3_referenced], ignore_index=True)

    # Candidates come from the normalized data root (TRAIN_S2/TRAIN_S3), so they
    # should already carry name_norm/address_norm/country_norm -- fall back to
    # normalizing this small, already-filtered-to-candidates subset in-memory if
    # they don't (mirrors Section 6's Source-1 fallback; never the full S2/S3 tables).
    missing_candidate_norm_cols = [c for c in NORMALIZED_COLUMNS if c not in candidate_lookup.columns]
    if missing_candidate_norm_cols and len(candidate_lookup):
        candidate_lookup = add_normalized_columns(candidate_lookup)

    feature_df = build_pair_feature_frame_normalized_chunked(
        long_pairs, s1_norm, candidate_lookup, chunksize=CANDIDATE_CHUNKSIZE,
    )

    print(f"Candidate rows fetched from Source-2/3   : {len(candidate_lookup):,}")
    print(f"Candidate pairs to featurize             : {len(long_pairs):,}")
    print(f"  (processed in chunks of {CANDIDATE_CHUNKSIZE:,} pairs)")
    print(f"Feature rows built                       : {len(feature_df):,}")
    display(feature_df.head(5))
else:
    print("No real candidate pairs yet -- skipping baseline feature generation.")

## 8. Entity-level train / validation split

Splits by S1 **entity**, not by candidate row. If candidate pairs were split independently,
some pairs for the same S1 entity could land in train while others land in validation -- the
model would then be validated on an entity whose sibling pairs it already trained on, leaking
information and inflating the validation score. `src.splitting.split_pairs_by_entity` guarantees
every pair for a given S1 lands entirely on one side.

In [ ]:
# =========================
# 8.1 Split
# =========================
from src.splitting import (
    entity_level_train_val_split,
    split_pairs_by_entity,
    assert_no_entity_leakage,
    DEFAULT_VALIDATION_FRACTION,
)

train_features = val_features = train_labels = val_labels = None

if CANDIDATES_AVAILABLE and feature_df is not None and label_df is not None:
    train_ids, val_ids = entity_level_train_val_split(s1["entity_id"])
    assert_no_entity_leakage(train_ids, val_ids)

    _train_features, _val_features = split_pairs_by_entity(feature_df, train_ids, val_ids)
    _train_labels, _val_labels = split_pairs_by_entity(label_df, train_ids, val_ids)

    print(f"Train S1 entities      : {len(train_ids):,}")
    print(f"Validation S1 entities : {len(val_ids):,}")
    print(f"Train pairs            : {len(_train_features):,}")
    print(f"Validation pairs       : {len(_val_features):,}")
    print("Entity leakage check   : PASSED (assert_no_entity_leakage raised nothing)")

    if len(_train_features) == 0 or len(_val_features) == 0:
        print(
            "Train or validation split has zero pairs (too few candidate pairs for "
            f"val_fraction={DEFAULT_VALIDATION_FRACTION} at the current data size) -- "
            "skipping model training, threshold sweep, and error analysis rather than "
            "training or scoring on an empty split. This resolves itself once the real, "
            "full-size candidate file is in use."
        )
    else:
        train_features, val_features = _train_features, _val_features
        train_labels, val_labels = _train_labels, _val_labels
else:
    print("No real candidate pairs/features yet -- skipping the train/validation split.")

## 9. Baseline model

Trains only when real candidate data (features + labels, entity-split) exists. Uses
`src.model.default_classifier` -- a placeholder classifier, **not** a claim that this is the
final model choice (see the docstring in `src/model.py`).

In [ ]:
# =========================
# 9.1 Train the baseline classifier
# =========================
from src.model import assemble_training_data, train_classifier, predict_match_scores, BASELINE_FEATURE_COLUMNS

model = None
val_scores = None

if train_features is not None and train_labels is not None:
    training_df = assemble_training_data(train_features, train_labels)
    model = train_classifier(training_df)

    pos = int((training_df["label"] == 1).sum())
    neg = int((training_df["label"] == 0).sum())

    print(f"Training rows      : {len(training_df):,}")
    print(f"Positive / Negative: {pos:,} / {neg:,}")
    print(f"Feature columns    : {BASELINE_FEATURE_COLUMNS}")
    print(f"Model              : {model}")

    val_scores = predict_match_scores(model, val_features)
    print(f"Validation rows scored: {len(val_scores):,}")
    print("(Full validation performance is reported in Part 10 -- Threshold Sweep.)")
else:
    print("No real candidate data yet -- skipping baseline model training.")

## 10. Threshold sweep

Uses `src.evaluation.threshold_sweep` with the standardized `(actual, predicted)` argument
order. Selected **only on the validation split** -- the hidden test set is never used for
threshold tuning.

In [ ]:
# =========================
# 10.1 Sweep thresholds on validation only
# =========================
from src.evaluation import threshold_sweep

sweep = None
best_threshold = None

if val_scores is not None and val_labels is not None:
    val_actual_by_s1 = {
        s1_id: set(group.loc[group["label"] == 1, "candidate_entity_id"])
        for s1_id, group in val_labels.groupby("source1_entity_id")
    }

    sweep = threshold_sweep(val_scores, val_actual_by_s1)
    best_row = sweep.sort_values("macro_f05", ascending=False).iloc[0]
    best_threshold = float(best_row["threshold"])

    print("Best validation threshold:")
    print(best_row.to_frame("value"))
    print()
    print("Full sweep:")
    display(sweep)
else:
    print("No validation scores yet -- skipping threshold sweep.")

## 11. Error analysis

Uses `src.error_analysis` (not re-implemented here) to assemble a wide inspection table and
surface representative hard cases at the selected threshold.

In [ ]:
# =========================
# 11.1 Build the error table at the selected threshold
# =========================
from src.error_analysis import build_error_table, filter_outcome

error_table = None

if val_scores is not None and val_labels is not None and best_threshold is not None:
    error_table = build_error_table(
        val_scores,
        val_labels,
        threshold=best_threshold,
        feature_df=val_features,
        source1_lookup=s1,
        candidate_lookup=candidate_lookup,
    )

    print("Outcome counts at threshold", best_threshold, ":")
    display(error_table["outcome"].value_counts().to_frame("count"))

    print("\nMost confident false positives:")
    display(filter_outcome(error_table, "FP", n=5))

    print("\nMost confident false negatives (model most wrongly certain a match was missing):")
    display(filter_outcome(error_table, "FN", n=5, ascending=True))
else:
    print("No validation predictions yet -- skipping error analysis.")

## 12. Experiment logging

Appends one row to the append-only experiment log (`src.experiment_log`) -- never overwrites
prior experiments.

In [ ]:
# =========================
# 12.1 Log this run
# =========================
from datetime import datetime, timezone
from src.experiment_log import ExperimentRecord, append_experiment, load_experiment_log

if model is not None and sweep is not None and candidate_report is not None:
    best_row = sweep.sort_values("macro_f05", ascending=False).iloc[0]
    record = ExperimentRecord(
        experiment_id=f"baseline_{datetime.now(timezone.utc):%Y%m%dT%H%M%S}",
        features_version="baseline_v1_B1-B6_normalized",
        model=type(model).__name__,
        model_params=model.get_params() if hasattr(model, "get_params") else {},
        threshold=best_threshold,
        candidate_pairs=len(label_df) if label_df is not None else None,
        positive_count=int((label_df["label"] == 1).sum()) if label_df is not None else None,
        negative_count=int((label_df["label"] == 0).sum()) if label_df is not None else None,
        candidate_recall=candidate_report["candidate_recall"],
        precision=float(best_row["global_precision"]),
        recall=float(best_row["global_recall"]),
        macro_f05=float(best_row["macro_f05"]),
        notes="Baseline pipeline run from the main notebook (pre-normalized, chunked features).",
    )
    append_experiment(record, EXPERIMENT_LOG_PATH)
    print(f"Logged experiment: {record.experiment_id}")
    display(load_experiment_log(EXPERIMENT_LOG_PATH).tail(5))
else:
    print("Nothing trained/evaluated yet -- skipping experiment logging.")

## 13. Test inference

Only runs once the test-set candidate pairs actually exist -- candidates are never fabricated.
Reuses the same chunked-loading, feature/labeling/scoring path already exercised on training data
above (`_load_candidate_pairs_long` from Part 5, `_load_referenced_rows` and the pre-normalized,
chunked feature builder from Part 7), so test-time features are generated by exactly the same
code path as training-time features -- no separate, potentially-drifting implementation.

In [ ]:
# =========================
# 13.1 Score the real test candidate pairs, if available
#      (same pre-normalized + chunked feature path as Part 7)
# =========================
TEST_CANDIDATES_AVAILABLE = TEST_CANDIDATE_PATH.exists() and TEST_S1.exists()

test_scores = None
test_predicted_by_s1 = None
test_candidate_by_s1 = None
test_source1 = None

if model is None:
    print("No trained model yet -- skipping test inference.")
elif not TEST_CANDIDATES_AVAILABLE:
    print(
        "WAITING FOR TEST-SET CANDIDATE PAIRS.\n"
        f"Expected at: {TEST_CANDIDATE_PATH}\n"
        "Skipping test inference and submission generation."
    )
else:
    from src.labeling import group_pairs_by_s1
    from src.evaluation import predictions_from_scores
    from src.normalization import add_normalized_columns
    from src.features import build_pair_feature_frame_normalized_chunked

    test_source1 = read_tsv(TEST_S1, dtype=str, keep_default_na=False)

    # Same normalized-column fallback as Section 6, applied to test Source-1.
    missing_test_s1_norm_cols = [c for c in NORMALIZED_COLUMNS if c not in test_source1.columns]
    if missing_test_s1_norm_cols:
        test_source1 = add_normalized_columns(test_source1)

    test_long_pairs = _load_candidate_pairs_long(TEST_CANDIDATE_PATH)
    test_candidate_by_s1 = group_pairs_by_s1(test_long_pairs)

    needed_test_ids = set(test_long_pairs["candidate_entity_id"])
    test_candidate_lookup = pd.concat(
        [
            _load_referenced_rows(TEST_S2, needed_test_ids),
            _load_referenced_rows(TEST_S3, needed_test_ids),
        ],
        ignore_index=True,
    )

    # Same normalized-column fallback as Part 7, applied to the (small,
    # already-filtered-to-candidates) test candidate lookup.
    missing_test_cand_norm_cols = [c for c in NORMALIZED_COLUMNS if c not in test_candidate_lookup.columns]
    if missing_test_cand_norm_cols and len(test_candidate_lookup):
        test_candidate_lookup = add_normalized_columns(test_candidate_lookup)

    test_feature_df = build_pair_feature_frame_normalized_chunked(
        test_long_pairs, test_source1, test_candidate_lookup, chunksize=CANDIDATE_CHUNKSIZE,
    )

    if len(test_feature_df) == 0:
        print("No test feature rows were built from the test candidate pairs -- skipping scoring.")
    else:
        test_scores = predict_match_scores(model, test_feature_df)

        threshold_for_test = best_threshold if best_threshold is not None else 0.5
        test_predicted_by_s1 = predictions_from_scores(test_scores, threshold_for_test)

        print(f"Test candidate pairs scored: {len(test_scores):,}")
        print(f"Threshold used              : {threshold_for_test}")

## 14. Submission generation

Uses the official column layout (`src.config.CANDIDATE_COLUMNS` / `MATCHING_COLUMNS`) via
`src.pipeline.make_submission_frames` / `write_submission` -- no invented schema. The organizer's
own validator (`student_resource/utils/validate_submission.py`) is loaded **from its original
location** and run as-is; it is never copied into this repo.

In [ ]:
# =========================
# 14.1 Write matching_results.tsv + candidate_pairs.tsv, then validate
# =========================
from src.pipeline import make_submission_frames, write_submission

if test_scores is None or test_predicted_by_s1 is None or test_candidate_by_s1 is None:
    print("No test predictions yet -- skipping submission generation.")
else:
    matching_df, candidates_df = make_submission_frames(
        test_source1, test_predicted_by_s1, test_candidate_by_s1
    )
    matching_path, candidate_path = write_submission(matching_df, candidates_df, OUTPUT_DIR)
    print(f"Wrote: {matching_path}")
    print(f"Wrote: {candidate_path}")

    if VALIDATOR_PATH.exists():
        import importlib.util

        spec = importlib.util.spec_from_file_location("official_validate_submission", VALIDATOR_PATH)
        official_validator = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(official_validator)

        print("\nRunning the official validator from:", VALIDATOR_PATH)
        # The organizer's module decides its own entry point (e.g. a `main()`
        # or `validate(...)` function) -- inspect `official_validator` for it
        # once the real file is available; this notebook does not guess or
        # reimplement its interface.
        print([name for name in dir(official_validator) if not name.startswith("_")])
    else:
        print(f"\nOfficial validator not found at {VALIDATOR_PATH} -- skipping validation step.")

## 15. Final checklist

- [ ] train normalization validated
- [ ] test normalization validated
- [ ] ground truth identity check = 1.0
- [ ] candidate file received
- [ ] candidate schema validated
- [ ] candidate recall measured
- [ ] candidate labels generated
- [ ] baseline features generated (pre-normalized, chunked path)
- [ ] entity-level split verified
- [ ] baseline model trained
- [ ] threshold tuned on validation
- [ ] error analysis completed
- [ ] test predictions generated
- [ ] candidate_pairs.tsv created
- [ ] matching_results.tsv created
- [ ] official validator passes

**Open scalability item** (see Part 5 markdown): the exploded long-format candidate pairs are
still concatenated into one in-memory DataFrame. Feature generation itself (Part 7/13) no longer
has this problem -- it is chunked and persist-to-disk-capable
(`build_pair_feature_frame_normalized_chunked(..., persist_path=...)`) -- but the labeling stage's
long-pairs accumulation would be the next thing to chunk-and-persist if the real candidate file
turns out to be too large for this to hold in memory comfortably.